# Exoplanet Pipeline Kaggle Real-Data Run

Run this notebook top-to-bottom after editing the **Configuration** cell. It trains the tabular AI model from curated labels plus TESS FITS, runs the science FITS/TIC-CTL pipeline, generates validation artifacts when labels are available, and packages outputs for download.

TIC/CTL files are target catalogs, not light curves. A meaningful run needs actual TESS light-curve FITS files attached to Kaggle or downloadable from MAST.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

import pandas as pd

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

print('Kaggle input datasets:')
for path in sorted(INPUT.glob('*')):
    print(' -', path)

print('\nSample input files:')
for i, path in enumerate(sorted(INPUT.rglob('*'))):
    if path.is_file():
        print(path)
        if i >= 120:
            break

## Configuration

Edit this cell for your Kaggle dataset names and run size. Keep the output paths under `/kaggle/working`.

In [ ]:
# Repository source. Use GitHub when Kaggle internet is enabled.
REPO_URL = 'https://github.com/im-rajnandan/exoplanet-pipeline-upgraded.git'
REPO_BRANCH = 'main'

# If internet is disabled, upload the repo as a Kaggle dataset and set this path.
# Example: REPO_DATASET_DIR = Path('/kaggle/input/exoplanet-pipeline-upgraded')
REPO_DATASET_DIR = None

# Curated labels. Required for supervised tabular training.
CURATED_CSV = Path('/kaggle/input/curated/curated_labels.csv')
CURATED_SOURCE = None  # set to 'tess-toi' when using downloaded public TOI metadata
LABEL_COL = 'label'

# Local TESS FITS used for training feature extraction. If not present and internet is enabled,
# set USE_MAST_DOWNLOADS=True and the notebook will download into /kaggle/working/data/lightcurves.
LIGHTCURVE_DIR = Path('/kaggle/input/tess-lightcurves')
USE_MAST_DOWNLOADS = False

# Science FITS directory. Use this when the science light curves are attached as a Kaggle dataset.
SCIENCE_FITS_DIR = Path('/kaggle/input/science-fits')

# TIC/CTL fallback. Use this only if you want to select targets from TIC/CTL and download light curves.
RUN_TIC_CTL_DOWNLOAD = False
TIC_CTL_CSV = Path('/kaggle/input/tic-ctl/exo_CTL_08.01.csv')
TIC_CTL_CATALOG_TYPE = 'ctl'  # one of: ctl, tic, xctl, target-list
TIC_CTL_MAX_TARGETS = 100
TESS_SECTOR = None  # set an int such as 1 to restrict MAST downloads, or None for any sector

# Epoch handling. Use 2457000 for BJD-like TESS epochs. Set None if labels already use BTJD.
EPOCH_OFFSET = 2457000

# Run sizes. Start small; increase after manifests look sane.
MAX_LABEL_ROWS = None  # e.g. 200 for a first training smoke run
N_PERIODS_FEATURES = 3000
N_PERIODS_SCIENCE = 4000
N_WORKERS_FEATURES = 4
N_WORKERS_SCIENCE = 2
N_BOOTSTRAP_FEATURES = 80
FEATURE_CHECKPOINT_EVERY = 5
TIMEOUT_SECONDS = 300
NO_VARIANTS = True

# Optional CNN. This is slower and needs the deep extra plus GPU for training speed.
RUN_OPTIONAL_CNN = False
CNN_METADATA_TOP = 2000
CNN_MAX_ROWS = 2000
CNN_EPOCHS = 20
CNN_BATCH_SIZE = 32

# Optional validation. Set this to a final candidate catalog that already has labels joined in.
# The notebook does not auto-join labels because one TIC can produce multiple candidates.
LABELED_SCIENCE_CATALOG = None

# Output paths.
REPO = WORK / 'exoplanet-pipeline'
OUT_LABELS = WORK / 'outputs_labeled_candidates'
OUT_AI = WORK / 'outputs_part6_ai'
OUT_CNN = WORK / 'outputs_cnn'
OUT_SCIENCE = WORK / 'outputs_science'
OUT_VALIDATION = WORK / 'outputs_validation'
LIGHTCURVE_DOWNLOAD_DIR = WORK / 'data' / 'lightcurves'

print('Configuration loaded.')

In [ ]:
def run_cmd(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if check and returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}: {cmd}')
    return subprocess.CompletedProcess(cmd, returncode, output, None)

def path_has_fits(path):
    path = Path(path)
    return path.exists() and any(path.rglob('*.fits')) or (path.exists() and any(path.rglob('*.fits.gz')))

def assert_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')
    print(f'{label}: {path}')

print('Helpers ready.')

## Set Up The Repository

In [ ]:
if REPO.exists():
    print(f'Repo already exists: {REPO}')
else:
    if REPO_DATASET_DIR is not None and Path(REPO_DATASET_DIR).exists():
        print(f'Copying repo dataset from {REPO_DATASET_DIR}')
        shutil.copytree(Path(REPO_DATASET_DIR), REPO)
    else:
        run_cmd(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], cwd=WORK)

run_cmd(['git', 'status', '--short', '--branch'], cwd=REPO)
run_cmd(['git', 'log', '--oneline', '-3'], cwd=REPO)

## Install Dependencies

This installs the repo in editable mode. The `deep` extra is included so the optional CNN path works; remove it if you want a lighter install.

In [ ]:
run_cmd([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,mast,deep]'], cwd=REPO)

## Validate Inputs

In [ ]:
assert_exists(CURATED_CSV, 'curated label CSV')

if LIGHTCURVE_DIR.exists():
    print('training light-curve dir:', LIGHTCURVE_DIR)
    print('training FITS sample count:', sum(1 for _ in LIGHTCURVE_DIR.rglob('*.fits')) + sum(1 for _ in LIGHTCURVE_DIR.rglob('*.fits.gz')))
elif USE_MAST_DOWNLOADS:
    print('Training FITS directory not attached; will download missing light curves to', LIGHTCURVE_DOWNLOAD_DIR)
else:
    raise FileNotFoundError(f'LIGHTCURVE_DIR does not exist and USE_MAST_DOWNLOADS=False: {LIGHTCURVE_DIR}')

if SCIENCE_FITS_DIR.exists():
    print('science FITS dir:', SCIENCE_FITS_DIR)
    print('science FITS sample count:', sum(1 for _ in SCIENCE_FITS_DIR.rglob('*.fits')) + sum(1 for _ in SCIENCE_FITS_DIR.rglob('*.fits.gz')))
elif RUN_TIC_CTL_DOWNLOAD:
    assert_exists(TIC_CTL_CSV, 'TIC/CTL catalog CSV')
else:
    print('No science FITS directory found. Set SCIENCE_FITS_DIR or enable RUN_TIC_CTL_DOWNLOAD.')

## Smoke Test

In [ ]:
run_cmd([
    sys.executable, 'scripts/run_parts_9_10_synthetic_batch.py',
    '--output-dir', str(WORK / 'smoke_outputs'),
    '--n-periods', '500',
], cwd=REPO)

## Inspect Curated Labels

In [ ]:
labels = pd.read_csv(CURATED_CSV)
print('shape:', labels.shape)
print('columns:', labels.columns.tolist())
display(labels.head())

if LABEL_COL not in labels.columns:
    raise ValueError(f'LABEL_COL={LABEL_COL!r} not found. Available columns: {labels.columns.tolist()}')
print(labels[LABEL_COL].value_counts(dropna=False))

## Build Labeled Candidate Feature Catalog

This is the critical bridge from curated TIC labels to the tabular AI model. If the manifest shows many `missing_lightcurve` rows, fix the FITS path or enable MAST downloads.

In [ ]:
feature_lightcurve_dir = LIGHTCURVE_DIR if LIGHTCURVE_DIR.exists() else LIGHTCURVE_DOWNLOAD_DIR

cmd = [
    sys.executable, 'scripts/build_labeled_candidate_catalog.py',
    str(CURATED_CSV),
    '--label-col', LABEL_COL,
    '--lightcurve-dir', str(feature_lightcurve_dir),
    '--output-dir', str(OUT_LABELS),
    '--n-periods', str(N_PERIODS_FEATURES),
    '--n-workers', str(N_WORKERS_FEATURES),
    '--n-bootstrap', str(N_BOOTSTRAP_FEATURES),
    '--checkpoint-every', str(FEATURE_CHECKPOINT_EVERY),
    '--progress-every', '1',
]
if CURATED_SOURCE is not None:
    cmd.extend(['--source', CURATED_SOURCE])
if USE_MAST_DOWNLOADS and not LIGHTCURVE_DIR.exists():
    cmd.append('--download-missing')
if EPOCH_OFFSET is not None:
    cmd.extend(['--epoch-offset', str(EPOCH_OFFSET)])
if MAX_LABEL_ROWS is not None:
    cmd.extend(['--max-rows', str(MAX_LABEL_ROWS)])
if NO_VARIANTS:
    cmd.append('--no-variants')

run_cmd(cmd, cwd=REPO)

In [ ]:
features_path = OUT_LABELS / 'labeled_candidate_features.csv'
manifest_path = OUT_LABELS / 'labeled_candidate_manifest.csv'
summary_path = OUT_LABELS / 'labeled_candidate_summary.json'

features = pd.read_csv(features_path)
manifest = pd.read_csv(manifest_path)
summary = json.loads(summary_path.read_text())

print(json.dumps(summary, indent=2))
print('\nFeature shape:', features.shape)
display(features.head())
print('\nLabel counts:')
print(features['label'].value_counts(dropna=False) if 'label' in features.columns else 'No label column')
print('\nManifest statuses:')
print(manifest['status'].value_counts(dropna=False))
display(manifest.head())

if features.empty:
    raise RuntimeError('No labeled feature rows were built. Check FITS paths, TIC IDs, labels, and manifest statuses before training.')

## Train Tabular AI Model

In [ ]:
run_cmd([
    sys.executable, 'scripts/train_ai_classifier_from_catalog.py',
    str(features_path),
    '--label-col', 'label',
    '--output-dir', str(OUT_AI),
    '--model-type', 'random_forest',
], cwd=REPO)

In [ ]:
metrics_path = OUT_AI / 'part6_ai_classifier_metrics.json'
metrics = json.loads(metrics_path.read_text())

print('classes:', metrics['class_names'])
print('training metadata:', metrics['training_metadata'])
print('\ntest metrics:')
for key, value in metrics['test_metrics'].items():
    if key != 'classification_report':
        print(key, value)

display(pd.read_csv(OUT_AI / 'part6_ai_classifier_feature_importance.csv').head(20))

## Optional CNN Training

Leave `RUN_OPTIONAL_CNN=False` for the first real run. The tabular model is the main supervised model for the curated dataset.

In [ ]:
if RUN_OPTIONAL_CNN:
    run_cmd([
        sys.executable, 'scripts/download_public_metadata.py',
        '--source', 'tess-toi',
        '--output-dir', str(WORK / 'data' / 'public'),
        '--top', str(CNN_METADATA_TOP),
    ], cwd=REPO)
    run_cmd([
        sys.executable, 'scripts/build_public_cnn_examples.py',
        str(WORK / 'data' / 'public' / 'metadata' / 'tess-toi_normalized_metadata.csv'),
        '--output-dir', str(WORK / 'data' / 'public' / 'cnn_examples'),
        '--lightcurve-dir', str(WORK / 'data' / 'public' / 'lightcurves'),
        '--download-missing',
        '--max-rows', str(CNN_MAX_ROWS),
        '--n-workers', '8',
    ], cwd=REPO)
    run_cmd([
        sys.executable, 'scripts/train_cnn_classifier.py',
        str(WORK / 'data' / 'public' / 'cnn_examples'),
        '--output-dir', str(OUT_CNN),
        '--epochs', str(CNN_EPOCHS),
        '--batch-size', str(CNN_BATCH_SIZE),
        '--device', 'cuda',
    ], cwd=REPO)
else:
    print('Skipping optional CNN training.')

## Run On Science Data

In [ ]:
ai_model = OUT_AI / 'part6_ai_classifier.joblib'
assert_exists(ai_model, 'trained AI model')

cnn_model_args = []
if RUN_OPTIONAL_CNN and (OUT_CNN / 'cnn_model.pt').exists():
    cnn_model_args = ['--cnn-model', str(OUT_CNN)]

if SCIENCE_FITS_DIR.exists():
    science_cmd = [
        sys.executable, 'scripts/run_parts_9_10_fits_directory.py',
        str(SCIENCE_FITS_DIR),
        '--output-dir', str(OUT_SCIENCE),
        '--ai-model', str(ai_model),
        '--n-workers', str(N_WORKERS_SCIENCE),
        '--n-periods', str(N_PERIODS_SCIENCE),
        '--timeout-seconds', str(TIMEOUT_SECONDS),
        *cnn_model_args,
    ]
    if NO_VARIANTS:
        science_cmd.append('--no-variants')
    run_cmd(science_cmd, cwd=REPO)
elif RUN_TIC_CTL_DOWNLOAD:
    target_list = WORK / 'data' / 'tic_ctl_targets.csv'
    target_list.parent.mkdir(parents=True, exist_ok=True)
    run_cmd([
        sys.executable, 'scripts/build_tic_ctl_target_list.py',
        str(TIC_CTL_CSV),
        '--catalog-type', TIC_CTL_CATALOG_TYPE,
        '--max-targets', str(TIC_CTL_MAX_TARGETS),
        '--output-csv', str(target_list),
    ], cwd=REPO)
    tic_cmd = [
        sys.executable, 'scripts/run_tic_ctl_pipeline.py',
        str(target_list),
        '--catalog-type', 'target-list',
        '--download-dir', str(LIGHTCURVE_DOWNLOAD_DIR),
        '--output-dir', str(OUT_SCIENCE),
        '--max-targets', str(TIC_CTL_MAX_TARGETS),
        '--ai-model', str(ai_model),
        '--n-workers', str(N_WORKERS_SCIENCE),
        '--n-periods', str(N_PERIODS_SCIENCE),
        '--timeout-seconds', str(TIMEOUT_SECONDS),
        *cnn_model_args,
    ]
    if TESS_SECTOR is not None:
        tic_cmd.extend(['--sector', str(TESS_SECTOR)])
    run_cmd(tic_cmd, cwd=REPO)
else:
    raise FileNotFoundError('No science FITS directory found and RUN_TIC_CTL_DOWNLOAD=False.')

## Inspect Science Outputs

In [ ]:
catalog_path = OUT_SCIENCE / 'batch_final_candidate_catalog.csv'
summary_path = OUT_SCIENCE / 'batch_target_summary.csv'
failure_path = OUT_SCIENCE / 'batch_failure_log.csv'

catalog = pd.read_csv(catalog_path) if catalog_path.exists() else pd.DataFrame()
target_summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
failures = pd.read_csv(failure_path) if failure_path.exists() else pd.DataFrame()

print('final catalog:', catalog.shape)
display(catalog.head())
if 'final_predicted_class' in catalog.columns:
    print(catalog['final_predicted_class'].value_counts(dropna=False))

print('\ntarget summary:', target_summary.shape)
if 'status' in target_summary.columns:
    print(target_summary['status'].value_counts(dropna=False))
display(target_summary.head())

print('\nfailures:', failures.shape)
display(failures.head())

## Optional Validation

In [ ]:
validation_report = OUT_VALIDATION / 'validation_report.json'

if LABELED_SCIENCE_CATALOG is not None and Path(LABELED_SCIENCE_CATALOG).exists():
    run_cmd([
        sys.executable, 'scripts/validate_candidate_catalog.py',
        '--catalog', str(LABELED_SCIENCE_CATALOG),
        '--label-col', LABEL_COL,
        '--pred-col', 'final_predicted_class',
        '--out-dir', str(OUT_VALIDATION),
    ], cwd=REPO)
else:
    print('Skipping validation. Set LABELED_SCIENCE_CATALOG to a final candidate catalog with labels joined in.')
    validation_report = None

## Generate Final Assets

In [ ]:
asset_cmd = [
    sys.executable, 'scripts/generate_final_submission_assets.py',
    str(catalog_path),
    '--output-dir', str(OUT_SCIENCE / 'submission_assets'),
]
if validation_report is not None and Path(validation_report).exists():
    asset_cmd.extend(['--validation-report', str(validation_report)])

run_cmd(asset_cmd, cwd=REPO)

print('Submission assets:')
for path in sorted((OUT_SCIENCE / 'submission_assets').glob('*')):
    print(path)

## Package Outputs

In [ ]:
zip_path = WORK / 'exoplanet_outputs.zip'
if zip_path.exists():
    zip_path.unlink()

items_to_zip = [
    OUT_LABELS,
    OUT_AI,
    OUT_CNN if OUT_CNN.exists() else None,
    OUT_SCIENCE,
    OUT_VALIDATION if OUT_VALIDATION.exists() else None,
    WORK / 'smoke_outputs',
]
items_to_zip = [p for p in items_to_zip if p is not None and Path(p).exists()]

run_cmd(['zip', '-r', str(zip_path), *[str(p.relative_to(WORK)) for p in items_to_zip]], cwd=WORK)
print('Packaged:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1024 / 1024, 2))

## What To Check Before Submission

- `outputs_labeled_candidates/labeled_candidate_manifest.csv`: low `missing_lightcurve` and `feature_failed` counts.
- `outputs_part6_ai/part6_ai_classifier_metrics.json`: holdout metrics and class counts are credible.
- `outputs_science/batch_target_summary.csv`: most targets are not failed/timeouts.
- `outputs_science/batch_final_candidate_catalog.csv`: contains period, duration, depth, confidence, and final class columns.
- `outputs_science/submission_assets/submission_three_page_report_draft.md`: edit this into the final 3-page report with exact data sources and validation numbers.